<a href="https://colab.research.google.com/github/mloulaps-rgb/---/blob/main/HCI_%D0%9B%D0%A07_%D0%90%D0%B2%D1%82%D0%BE%D0%B7%D0%B0%D0%B2%D0%B5%D1%80%D1%88%D0%B5%D0%BD%D0%BD%D1%8F_%D1%82%D0%B5%D0%BA%D1%81%D1%82%D1%83_N_%D0%B3%D1%80%D0%B0%D0%BC%D0%B8_%D0%92%D0%B5%D0%BB%D0%B8%D1%87%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Крок 1. Завантаження та попередня обробка корпусу
import os, re, random
import numpy as np
import nltk
from collections import Counter
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Завантаження корпусу твітів (англомовний корпус курсу NLP)
URL = ("https://raw.githubusercontent.com/amanchadha/"
       "coursera-natural-language-processing-specialization/master/"
       "2%20-%20Natural%20Language%20Processing%20with%20Probabilistic%20Models/"
       "Week%203/en_US.twitter.txt")
if not os.path.exists("en_US.twitter.txt"):
    os.system(f'wget -q "{URL}" -O en_US.twitter.txt')

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def split_to_sentences(data):
    sentences = [s.strip() for s in data.split("\n")]
    return [s for s in sentences if len(s) > 0]

def tokenize_sentences(sentences):
    tokenized = []
    for sentence in sentences:
        tokenized.append(nltk.word_tokenize(sentence.lower()))
    return tokenized

def get_tokenized_data(data):
    return tokenize_sentences(split_to_sentences(data))

def count_words(tokenized_sentences):
    word_counts = {}
    for sentence in tokenized_sentences:
        for token in sentence:
            word_counts[token] = word_counts.get(token, 0) + 1
    return word_counts

def get_words_with_nplus_frequency(tokenized_sentences, count_threshold):
    word_counts = count_words(tokenized_sentences)
    return [w for w, c in word_counts.items() if c >= count_threshold]

def replace_oov_words_by_unk(tokenized_sentences, vocabulary, unknown_token="<unk>"):
    vocabulary = set(vocabulary)
    return [[t if t in vocabulary else unknown_token for t in s] for s in tokenized_sentences]

def preprocess_data(train_data, test_data, count_threshold):
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)
    train_r = replace_oov_words_by_unk(train_data, vocabulary)
    test_r  = replace_oov_words_by_unk(test_data, vocabulary)
    return train_r, test_r, vocabulary

# Завантаження та токенізація
data = load_data("en_US.twitter.txt")
tokenized_data = get_tokenized_data(data)

# Поділ на тренувальну (80%) та тестову (20%) вибірки
random.seed(42)
random.shuffle(tokenized_data)
train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[:train_size]
test_data  = tokenized_data[train_size:]

count_threshold = 2
train_data_processed, test_data_processed, vocabulary = preprocess_data(train_data, test_data, count_threshold)

print("Усього речень у корпусі :", len(tokenized_data))
print("Тренувальна вибірка     :", len(train_data), "речень")
print("Тестова вибірка         :", len(test_data), "речень")
print("Розмір словника (>=2)    :", len(vocabulary), "слів")
print("Приклад обробленого речення:", train_data_processed[0][:12])


Усього речень у корпусі : 47961
Тренувальна вибірка     : 38368 речень
Тестова вибірка         : 9593 речень
Розмір словника (>=2)    : 14758 слів
Приклад обробленого речення: ['what', 'little', 'i', 'have', 'goes', 'toward', 'my', 'commute', '.', '$', '4', 'gasoline']


In [2]:
# Крок 2. Побудова N-грамної моделі (від уніграм до 5-грам)
def count_n_grams(data, n, start_token="<s>", end_token="</s>"):
    """Підраховує кількість n-грам у даних."""
    n_grams = {}
    for sentence in data:
        sentence = [start_token] * (n - 1) + sentence + [end_token]
        sentence = tuple(sentence)
        for i in range(len(sentence) - n + 1):
            n_gram = sentence[i:i + n]
            n_grams[n_gram] = n_grams.get(n_gram, 0) + 1
    return n_grams

# Будуємо моделі від уніграм (n=1) до 5-грам (n=5)
n_gram_counts_list = []
for n in range(1, 6):
    counts = count_n_grams(train_data_processed, n)
    n_gram_counts_list.append(counts)
    print(f"{n}-грам: {len(counts):>7} унікальних")

print("\nПриклади біграм:")
for bg in [("i", "love"), ("thank", "you"), ("<s>", "i")]:
    print(f"  {bg} -> {n_gram_counts_list[1].get(bg, 0)}")


1-грам:   14760 унікальних
2-грам:  210064 унікальних
3-грам:  433683 унікальних
4-грам:  532259 унікальних
5-грам:  558746 унікальних

Приклади біграм:
  ('i', 'love') -> 555
  ('thank', 'you') -> 538
  ('<s>', 'i') -> 4128


In [3]:
# Крок 3. Обчислення ймовірностей наступного слова (k-згладжування)
def estimate_probability(word, previous_n_gram, n_gram_counts,
                         n_plus1_gram_counts, vocabulary_size, k=1.0):
    """Оцінює ймовірність слова після попередньої n-грами з add-k згладжуванням."""
    previous_n_gram = tuple(previous_n_gram)
    previous_n_gram_count = n_gram_counts.get(previous_n_gram, 0)
    denominator = previous_n_gram_count + k * vocabulary_size
    n_plus1_gram = previous_n_gram + (word,)
    n_plus1_gram_count = n_plus1_gram_counts.get(n_plus1_gram, 0)
    numerator = n_plus1_gram_count + k
    return numerator / denominator

def estimate_probabilities(previous_n_gram, n_gram_counts,
                           n_plus1_gram_counts, vocabulary, k=1.0):
    """Оцінює ймовірності всіх можливих наступних слів."""
    vocabulary = vocabulary + ["</s>", "<unk>"]
    vocabulary_size = len(vocabulary)
    probabilities = {}
    for word in vocabulary:
        probabilities[word] = estimate_probability(
            word, previous_n_gram, n_gram_counts,
            n_plus1_gram_counts, vocabulary_size, k)
    return probabilities

# Демонстрація: найімовірніші наступні слова після "i want to"
previous = ["i", "want", "to"]
probs = estimate_probabilities(previous, n_gram_counts_list[2], n_gram_counts_list[3], vocabulary, k=1.0)
top = sorted(((w, p) for w, p in probs.items() if w not in ("<unk>",)), key=lambda x: -x[1])[:7]
print(f"Найімовірніші слова після {previous}:")
for w, p in top:
    print(f"  {w:10s}  P = {p:.6f}")


Найімовірніші слова після ['i', 'want', 'to']:
  go          P = 0.001075
  be          P = 0.001008
  do          P = 0.000605
  get         P = 0.000403
  see         P = 0.000403
  live        P = 0.000403
  .           P = 0.000269


In [4]:
# Крок 4. Розрахунок перплексії (міра якості моделі)
def calculate_perplexity(sentence, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """Обчислює перплексію для речення."""
    n = len(list(n_gram_counts.keys())[0])
    sentence = ["<s>"] * n + sentence + ["</s>"]
    sentence = tuple(sentence)
    N = len(sentence)
    product_pi = 1.0
    for t in range(n, N):
        n_gram = sentence[t - n:t]
        word = sentence[t]
        probability = estimate_probability(word, n_gram, n_gram_counts,
                                           n_plus1_gram_counts, vocabulary_size, k)
        product_pi *= 1 / probability
    return product_pi ** (1 / N)

V = len(vocabulary) + 2  # +2 для </s> та <unk>

# Середня перплексія на підмножині тестових речень для біграмної та триграмної моделей
sample = [s for s in test_data_processed[:150] if len(s) > 0]
for name, (ngi, np1i) in [("Біграмна (n=2)", (1, 2)), ("Триграмна (n=3)", (2, 3))]:
    pps = [calculate_perplexity(s, n_gram_counts_list[ngi], n_gram_counts_list[np1i], V, k=1.0) for s in sample]
    print(f"{name}: середня перплексія = {np.mean(pps):.2f}  (на {len(pps)} реченнях)")

# Приклад для одного речення
demo = ["i", "am", "happy", "to", "see", "you"]
pp = calculate_perplexity(demo, n_gram_counts_list[1], n_gram_counts_list[2], V, k=1.0)
print(f"\nПерплексія речення {demo}: {pp:.2f}")


Біграмна (n=2): середня перплексія = 2244.43  (на 150 реченнях)
Триграмна (n=3): середня перплексія = 2309.67  (на 150 реченнях)

Перплексія речення ['i', 'am', 'happy', 'to', 'see', 'you']: 134.47


In [5]:
# Крок 5. Функція автозавершення тексту
def suggest_a_word(previous_tokens, n_gram_counts, n_plus1_gram_counts,
                   vocabulary, k=1.0, start_with=None):
    """Пропонує найімовірніше наступне слово."""
    n = len(list(n_gram_counts.keys())[0])
    previous_tokens = ["<s>"] * n + previous_tokens
    previous_n_gram = previous_tokens[-n:]
    probabilities = estimate_probabilities(previous_n_gram, n_gram_counts,
                                           n_plus1_gram_counts, vocabulary, k)
    suggestion = None
    max_prob = 0
    for word, prob in probabilities.items():
        if start_with and not word.startswith(start_with):
            continue
        if prob > max_prob:
            suggestion = word
            max_prob = prob
    return suggestion, max_prob

def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None, top_n=5):
    """Пропозиції від усіх N-грамних моделей одразу."""
    suggestions = []
    for i in range(len(n_gram_counts_list) - 1):
        n_gram_counts = n_gram_counts_list[i]
        n_plus1_gram_counts = n_gram_counts_list[i + 1]
        ctx = previous_tokens[-i - 1:] if i < len(previous_tokens) else ["<s>"] * (i + 1 - len(previous_tokens)) + previous_tokens
        all_probs = estimate_probabilities(ctx, n_gram_counts, n_plus1_gram_counts, vocabulary, k)
        if start_with:
            all_probs = {w: p for w, p in all_probs.items() if w.startswith(start_with)}
        top = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)[:top_n]
        suggestions.append(top)
    return suggestions

# Демонстрація автозавершення для різних контекстів
for ctx in [["i", "am"], ["happy", "to"], ["how", "are"], ["i", "want", "to", "go"]]:
    sug, p = suggest_a_word(ctx, n_gram_counts_list[2], n_gram_counts_list[3], vocabulary, k=1.0)
    print(f"{str(ctx):35s} -> '{sug}'  (P = {p:.5f})")

# Автозавершення з фільтром за початковою літерою
print("\nФільтр за початком 'th' після ['i','want','to']:")
sug, p = suggest_a_word(["i", "want", "to"], n_gram_counts_list[2], n_gram_counts_list[3],
                        vocabulary, k=1.0, start_with="th")
print(f"  -> '{sug}'  (P = {p:.5f})")

# Пропозиції від усіх моделей (уніграми..5-грами)
print("\nПропозиції від усіх N-грамних моделей для ['i','love']:")
allsug = get_suggestions(["i", "love"], n_gram_counts_list, vocabulary, k=1.0, top_n=3)
for i, s in enumerate(allsug, start=1):
    clean = [(w, round(p, 5)) for w, p in s if w not in ("<unk>",)][:3]
    print(f"  {i}-грам модель: {clean}")


['i', 'am']                         -> 'so'  (P = 0.00127)
['happy', 'to']                     -> 'help'  (P = 0.00027)
['how', 'are']                      -> 'you'  (P = 0.00108)
['i', 'want', 'to', 'go']           -> 'to'  (P = 0.00095)

Фільтр за початком 'th' після ['i','want','to']:
  -> 'thank'  (P = 0.00013)

Пропозиції від усіх N-грамних моделей для ['i','love']:
  1-грам модель: [('you', 0.01397), ('it', 0.0082), ('to', 0.00638)]
  2-грам модель: [('you', 0.00803), ('it', 0.00353), ('the', 0.003)]
  3-грам модель: [('you', 0.00247), ('my', 0.00107), ('the', 0.00107)]
  4-грам модель: [('you', 0.00247), ('my', 0.00107), ('the', 0.00107)]


In [6]:
# Крок 6. Простий інтерфейс користувача (ipywidgets)
import ipywidgets as widgets
from IPython.display import display, clear_output

def create_ui(n_gram_counts_list, vocabulary):
    text_input = widgets.Text(value="", placeholder="Введіть текст...",
                              description="Текст:", layout=widgets.Layout(width="500px"))
    prefix_input = widgets.Text(value="", placeholder="Обмеження за першими літерами",
                                description="Початок:", layout=widgets.Layout(width="500px"))
    n_gram_dropdown = widgets.Dropdown(
        options=[("Уніграми", 1), ("Біграми", 2), ("Триграми", 3), ("4-грами", 4), ("5-грами", 5)],
        value=3, description="N-грами:")
    k_slider = widgets.FloatSlider(value=1.0, min=0.01, max=5.0, step=0.01,
                                   description="K:", continuous_update=False, readout_format=".2f")
    suggestion_output = widgets.Output()

    def on_button_click(word):
        cur = text_input.value
        if cur and not cur.endswith(" "):
            cur += " "
        text_input.value = cur + word + " "

    def update_suggestions(_):
        with suggestion_output:
            clear_output()
            cur = text_input.value.strip()
            if not cur:
                print("Введіть текст для отримання пропозицій."); return
            tokens = cur.lower().split()
            n = n_gram_dropdown.value
            start_with = prefix_input.value or None
            prev = tokens[-(n - 1):] if len(tokens) >= n - 1 else ["<s>"] * (n - 1 - len(tokens)) + tokens
            probs = estimate_probabilities(prev, n_gram_counts_list[n - 2], n_gram_counts_list[n - 1],
                                           vocabulary, k=k_slider.value) if n > 1 else \
                    estimate_probabilities([], n_gram_counts_list[0], n_gram_counts_list[1],
                                           vocabulary, k=k_slider.value)
            if start_with:
                probs = {w: p for w, p in probs.items() if w.startswith(start_with) and w not in ("<s>", "</s>", "<unk>")}
            else:
                probs = {w: p for w, p in probs.items() if w not in ("<s>", "</s>", "<unk>")}
            top = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5]
            if not top:
                print("Немає пропозицій для даного контексту."); return
            print(f"Пропозиції для '{cur}':")
            btns = []
            for word, prob in top:
                b = widgets.Button(description=f"{word} ({prob:.4f})")
                b.on_click(lambda x, w=word: on_button_click(w))
                btns.append(b)
            display(widgets.HBox(btns))

    suggest_button = widgets.Button(description="Запропонувати", button_style="primary")
    suggest_button.on_click(update_suggestions)
    display(text_input); display(prefix_input)
    display(widgets.HBox([n_gram_dropdown, k_slider]))
    display(suggest_button); display(suggestion_output)

create_ui(n_gram_counts_list, vocabulary)
print("Інтерфейс створено. Введіть текст і натисніть «Запропонувати».")


Text(value='', description='Текст:', layout=Layout(width='500px'), placeholder='Введіть текст...')

Text(value='', description='Початок:', layout=Layout(width='500px'), placeholder='Обмеження за першими літерам…

Button(button_style='primary', description='Запропонувати', style=ButtonStyle())

Output()

Інтерфейс створено. Введіть текст і натисніть «Запропонувати».


In [7]:
# Крок 7. Повний приклад: покрокове автозавершення фрази
def autocomplete_text(seed_tokens, steps=6, n=3, k=1.0):
    """Покрокове дописування тексту триграмною моделлю."""
    tokens = list(seed_tokens)
    for _ in range(steps):
        prev = tokens[-(n - 1):]
        sug, p = suggest_a_word(prev, n_gram_counts_list[n - 2], n_gram_counts_list[n - 1],
                                vocabulary, k=k)
        if sug in ("</s>", "<unk>", None):
            break
        tokens.append(sug)
    return tokens

for seed in [["i", "am"], ["thank", "you", "for"], ["can", "not", "wait"]]:
    result = autocomplete_text(seed, steps=6, n=3)
    print(f"{str(seed):28s} => {' '.join(result)}")

print("\nСистема автозавершення на основі N-грам працює коректно:")
print("- побудовано моделі від уніграм до 5-грам;")
print("- реалізовано згладжування add-k та метрику перплексії;")
print("- автозавершення пропонує контекстно доречні слова.")


['i', 'am']                  => i am so excited to see you there
['thank', 'you', 'for']      => thank you for the follow !
['can', 'not', 'wait']       => can not wait to see you there !

Система автозавершення на основі N-грам працює коректно:
- побудовано моделі від уніграм до 5-грам;
- реалізовано згладжування add-k та метрику перплексії;
- автозавершення пропонує контекстно доречні слова.
